**IMPORT**

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="lxjOP157H1bZgHOniuFX")
project = rf.workspace("laitsugas").project("final-aimedisina")
version = project.version(7)
roboflow_dataset = version.download(
    "coco",
    location="/kaggle/working/final-aimedisina-7"
)

print("Dataset location:", roboflow_dataset.location)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.8/336.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/final-aimedisina-7 in coco:: 100%|██████████| 5635/5635 [00:01<00:00, 4684.03it/s]


Dataset location: /kaggle/working/final-aimedisina-7


**DATASET**

In [2]:
import json
import os
import random

import torch
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class CocoSSDDataset(Dataset):
    r"""
    Reads a Roboflow COCO export:
        root/<split>/_annotations.coco.json
        root/<split>/<images>

    Returns:
        image:  (3, img_size, img_size) normalized tensor
        target: {"boxes":  (n, 4) x1y1x2y2 normalized to [0, 1],
                 "labels": (n,)   int64, 1..num_classes-1  (0 = background)}
    """

    def __init__(self, root, split, img_size=300, train=False):
        self.dir = os.path.join(root, split)
        self.img_size = img_size
        self.train = train

        with open(os.path.join(self.dir, "_annotations.coco.json")) as f:
            data = json.load(f)

        # Roboflow adds a dummy parent category (supercategory == "none"), drop it.
        cats = [c for c in data["categories"] if c.get("supercategory") != "none"]
        if not cats:
            cats = data["categories"]
        cats = sorted(cats, key=lambda c: c["id"])
        self.cat_id_to_label = {c["id"]: i + 1 for i, c in enumerate(cats)}
        self.class_names = ["__background__"] + [c["name"] for c in cats]
        self.num_classes = len(self.class_names)  # includes background

        self.images = sorted(data["images"], key=lambda im: im["id"])
        self.anns = {}
        for a in data["annotations"]:
            self.anns.setdefault(a["image_id"], []).append(a)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        info = self.images[idx]
        img = Image.open(os.path.join(self.dir, info["file_name"])).convert("RGB")
        W, H = img.size

        boxes, labels = [], []
        for a in self.anns.get(info["id"], []):
            if a["category_id"] not in self.cat_id_to_label:
                continue
            x, y, w, h = a["bbox"]  # COCO: x, y, w, h in pixels
            x1, y1 = max(0.0, x / W), max(0.0, y / H)
            x2, y2 = min(1.0, (x + w) / W), min(1.0, (y + h) / H)
            if (x2 - x1) < 1e-3 or (y2 - y1) < 1e-3:  # skip degenerate boxes
                continue
            boxes.append([x1, y1, x2, y2])
            labels.append(self.cat_id_to_label[a["category_id"]])

        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.tensor(labels, dtype=torch.int64)

        if self.train:
            # horizontal flip
            if random.random() < 0.5:
                img = TF.hflip(img)
                if boxes.numel():
                    boxes = torch.stack([1 - boxes[:, 2], boxes[:, 1],
                                         1 - boxes[:, 0], boxes[:, 3]], dim=1)
            # photometric jitter
            img = TF.adjust_brightness(img, random.uniform(0.75, 1.25))
            img = TF.adjust_contrast(img, random.uniform(0.75, 1.25))
            img = TF.adjust_saturation(img, random.uniform(0.75, 1.25))

        img = TF.resize(img, [self.img_size, self.img_size])
        img = TF.to_tensor(img)
        img = TF.normalize(img, IMAGENET_MEAN, IMAGENET_STD)

        return img, {"boxes": boxes, "labels": labels}


def collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images, 0), list(targets)

**MODEL**

In [3]:
import torch.nn as nn
import torch
import math
import torchvision


def get_iou(boxes1, boxes2):
    r"""
    IOU between two sets of boxes
    :param boxes1: (Tensor of shape N x 4)
    :param boxes2: (Tensor of shape M x 4)
    :return: IOU matrix of shape N x M
    """
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])  # (N,)
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])  # (M,)

    x_left = torch.max(boxes1[:, None, 0], boxes2[:, 0])  # (N, M)
    y_top = torch.max(boxes1[:, None, 1], boxes2[:, 1])  # (N, M)
    x_right = torch.min(boxes1[:, None, 2], boxes2[:, 2])  # (N, M)
    y_bottom = torch.min(boxes1[:, None, 3], boxes2[:, 3])  # (N, M)

    intersection_area = ((x_right - x_left).clamp(min=0) *
                         (y_bottom - y_top).clamp(min=0))  # (N, M)
    union = area1[:, None] + area2 - intersection_area  # (N, M)
    iou = intersection_area / union  # (N, M)
    return iou


def boxes_to_transformation_targets(ground_truth_boxes,
                                    default_boxes,
                                    weights=(10., 10., 5., 5.)):
    r"""
    Compute regression targets for each default box (x1y1x2y2 boxes).
    :param ground_truth_boxes: (Tensor of shape N x 4)
    :param default_boxes: (Tensor of shape N x 4)
    :return: regression_targets: (Tensor of shape N x 4)
    """
    widths = default_boxes[:, 2] - default_boxes[:, 0]
    heights = default_boxes[:, 3] - default_boxes[:, 1]
    center_x = default_boxes[:, 0] + 0.5 * widths
    center_y = default_boxes[:, 1] + 0.5 * heights

    gt_widths = (ground_truth_boxes[:, 2] - ground_truth_boxes[:, 0])
    gt_heights = ground_truth_boxes[:, 3] - ground_truth_boxes[:, 1]
    gt_center_x = ground_truth_boxes[:, 0] + 0.5 * gt_widths
    gt_center_y = ground_truth_boxes[:, 1] + 0.5 * gt_heights

    targets_dx = weights[0] * (gt_center_x - center_x) / widths
    targets_dy = weights[1] * (gt_center_y - center_y) / heights
    targets_dw = weights[2] * torch.log(gt_widths / widths)
    targets_dh = weights[3] * torch.log(gt_heights / heights)
    regression_targets = torch.stack((targets_dx,
                                      targets_dy,
                                      targets_dw,
                                      targets_dh), dim=1)
    return regression_targets


def apply_regression_pred_to_default_boxes(box_transform_pred,
                                           default_boxes,
                                           weights=(10., 10., 5., 5.)):
    r"""
    Transform default_boxes using predicted deltas (x1y1x2y2 boxes).
    :param box_transform_pred: (Tensor of shape N x 4)
    :param default_boxes: (Tensor of shape N x 4)
    :return: pred_boxes: (Tensor of shape N x 4)
    """
    w = default_boxes[:, 2] - default_boxes[:, 0]
    h = default_boxes[:, 3] - default_boxes[:, 1]
    center_x = default_boxes[:, 0] + 0.5 * w
    center_y = default_boxes[:, 1] + 0.5 * h

    dx = box_transform_pred[..., 0] / weights[0]
    dy = box_transform_pred[..., 1] / weights[1]
    dw = box_transform_pred[..., 2] / weights[2]
    dh = box_transform_pred[..., 3] / weights[3]

    pred_center_x = dx * w + center_x
    pred_center_y = dy * h + center_y
    pred_w = torch.exp(dw) * w
    pred_h = torch.exp(dh) * h

    pred_box_x1 = pred_center_x - 0.5 * pred_w
    pred_box_y1 = pred_center_y - 0.5 * pred_h
    pred_box_x2 = pred_center_x + 0.5 * pred_w
    pred_box_y2 = pred_center_y + 0.5 * pred_h

    pred_boxes = torch.stack((
        pred_box_x1,
        pred_box_y1,
        pred_box_x2,
        pred_box_y2),
        dim=-1)
    return pred_boxes


def generate_default_boxes(feat, aspect_ratios, scales):
    r"""
    Generate default boxes for all feature maps.
    :param feat: List[(Tensor of shape B x C x Feat_H x Feat_W)]
    :param aspect_ratios: List[List[float]] aspect ratios for each feature map
    :param scales: List[float] scales for each feature map
    :return: List[(Tensor of shape N x 4)] (x1y1x2y2, normalized), one per batch image
    """
    default_boxes = []
    for k in range(len(feat)):
        s_prime_k = math.sqrt(scales[k] * scales[k + 1])
        wh_pairs = [[s_prime_k, s_prime_k]]

        for ar in aspect_ratios[k]:
            sq_ar = math.sqrt(ar)
            w = scales[k] * sq_ar
            h = scales[k] / sq_ar
            wh_pairs.extend([[w, h]])

        feat_h, feat_w = feat[k].shape[-2:]

        shifts_x = ((torch.arange(0, feat_w) + 0.5) / feat_w).to(torch.float32)
        shifts_y = ((torch.arange(0, feat_h) + 0.5) / feat_h).to(torch.float32)
        shift_y, shift_x = torch.meshgrid(shifts_y, shifts_x, indexing="ij")
        shift_x = shift_x.reshape(-1)
        shift_y = shift_y.reshape(-1)

        shifts = torch.stack((shift_x, shift_y) * len(wh_pairs), dim=-1).reshape(-1, 2)

        wh_pairs = torch.as_tensor(wh_pairs)
        wh_pairs = wh_pairs.repeat((feat_h * feat_w), 1)

        default_box = torch.cat((shifts, wh_pairs), dim=1)
        default_boxes.append(default_box)
    default_boxes = torch.cat(default_boxes, dim=0)  # (8732, 4)

    dboxes = []
    for _ in range(feat[0].size(0)):
        dboxes_in_image = default_boxes
        dboxes_in_image = torch.cat(
            [
                (dboxes_in_image[:, :2] - 0.5 * dboxes_in_image[:, 2:]),
                (dboxes_in_image[:, :2] + 0.5 * dboxes_in_image[:, 2:]),
            ],
            -1,
        )
        dboxes.append(dboxes_in_image.to(feat[0].device))
    return dboxes


class SSD(nn.Module):
    r"""
    SSD300 with VGG16 backbone.

    Expects:
      - images: (B, 3, 300, 300), ImageNet-normalized
      - targets (training): list of dicts with
            "boxes":  (n, 4) x1y1x2y2 NORMALIZED to [0, 1]
            "labels": (n,)   int64, classes start at 1 (0 = background)
    Returns (losses, detections).
    """
    def __init__(self, config, num_classes=21):
        super().__init__()
        self.aspect_ratios = config['aspect_ratios']

        # copy so the config dict is not mutated
        self.scales = list(config['scales']) + [1.0]

        self.num_classes = num_classes
        self.iou_threshold = config['iou_threshold']
        self.low_score_threshold = config['low_score_threshold']
        self.neg_pos_ratio = config['neg_pos_ratio']
        self.pre_nms_topK = config['pre_nms_topK']
        self.nms_threshold = config['nms_threshold']
        self.detections_per_img = config['detections_per_img']

        # Load imagenet pretrained vgg network
        backbone = torchvision.models.vgg16(
            weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1
        )

        max_pool_pos = [idx for idx, layer in enumerate(list(backbone.features))
                        if isinstance(layer, nn.MaxPool2d)]
        max_pool_stage_3_pos = max_pool_pos[-3]  # 16
        max_pool_stage_4_pos = max_pool_pos[-2]  # 23

        backbone.features[max_pool_stage_3_pos].ceil_mode = True
        self.features = nn.Sequential(*backbone.features[:max_pool_stage_4_pos])
        self.scale_weight = nn.Parameter(torch.ones(512) * 20)

        # Conv5_3 + conv layers replacing fc6 and fc7
        fcs = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=3,
                      padding=6, dilation=6),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=1),
            nn.ReLU(inplace=True),
        )
        self.conv5_3_fc = nn.Sequential(
            *backbone.features[max_pool_stage_4_pos:-1],
            fcs,
        )

        # Additional conv layers
        self.conv8_2 = nn.Sequential(
            nn.Conv2d(1024, 256, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, kernel_size=3, padding=1, stride=2),
            nn.ReLU(inplace=True)
        )
        self.conv9_2 = nn.Sequential(
            nn.Conv2d(512, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=2),
            nn.ReLU(inplace=True)
        )
        self.conv10_2 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3),
            nn.ReLU(inplace=True)
        )
        self.conv11_2 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3),
            nn.ReLU(inplace=True)
        )

        # conv4_3, fcs, conv8_2, conv9_2, conv10_2, conv11_2
        out_channels = [512, 1024, 512, 256, 256, 256]

        # Prediction layers
        self.cls_heads = nn.ModuleList()
        for channels, aspect_ratio in zip(out_channels, self.aspect_ratios):
            self.cls_heads.append(nn.Conv2d(channels,
                                            self.num_classes * (len(aspect_ratio) + 1),
                                            kernel_size=3,
                                            padding=1))

        self.bbox_reg_heads = nn.ModuleList()
        for channels, aspect_ratio in zip(out_channels, self.aspect_ratios):
            self.bbox_reg_heads.append(nn.Conv2d(channels, 4 * (len(aspect_ratio) + 1),
                                                 kernel_size=3,
                                                 padding=1))

        # Conv layer initialization
        for layer in fcs.modules():
            if isinstance(layer, nn.Conv2d):
                torch.nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    torch.nn.init.constant_(layer.bias, 0.0)

        for conv_module in [self.conv8_2, self.conv9_2, self.conv10_2, self.conv11_2]:
            for layer in conv_module.modules():
                if isinstance(layer, nn.Conv2d):
                    torch.nn.init.xavier_uniform_(layer.weight)
                    if layer.bias is not None:
                        torch.nn.init.constant_(layer.bias, 0.0)

        for module in self.cls_heads:
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.0)
        for module in self.bbox_reg_heads:
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.0)

    def compute_loss(
            self,
            targets,
            cls_logits,
            bbox_regression,
            default_boxes,
            matched_idxs,
    ):
        num_foreground = 0
        bbox_loss = []
        cls_targets = []
        for (
            targets_per_image,
            bbox_regression_per_image,
            cls_logits_per_image,
            default_boxes_per_image,
            matched_idxs_per_image,
        ) in zip(targets, bbox_regression, cls_logits, default_boxes, matched_idxs):
            fg_idxs_per_image = torch.where(matched_idxs_per_image >= 0)[0]
            foreground_matched_idxs_per_image = matched_idxs_per_image[
                fg_idxs_per_image
            ]
            num_foreground += foreground_matched_idxs_per_image.numel()

            matched_gt_boxes_per_image = targets_per_image["boxes"][
                foreground_matched_idxs_per_image
            ]
            bbox_regression_per_image = bbox_regression_per_image[fg_idxs_per_image, :]
            default_boxes_per_image = default_boxes_per_image[fg_idxs_per_image, :]
            target_regression = boxes_to_transformation_targets(
                matched_gt_boxes_per_image,
                default_boxes_per_image)

            bbox_loss.append(
                torch.nn.functional.smooth_l1_loss(bbox_regression_per_image,
                                                   target_regression,
                                                   reduction='sum')
            )

            gt_classes_target = torch.zeros(
                (cls_logits_per_image.size(0),),
                dtype=targets_per_image["labels"].dtype,
                device=targets_per_image["labels"].device,
            )
            gt_classes_target[fg_idxs_per_image] = targets_per_image["labels"][
                foreground_matched_idxs_per_image
            ]
            cls_targets.append(gt_classes_target)

        bbox_loss = torch.stack(bbox_loss)
        cls_targets = torch.stack(cls_targets)  # (B, 8732)

        num_classes = cls_logits.size(-1)
        cls_loss = torch.nn.functional.cross_entropy(cls_logits.view(-1, num_classes),
                                                     cls_targets.view(-1),
                                                     reduction="none").view(
            cls_targets.size()
        )

        # Hard negative mining
        foreground_idxs = cls_targets > 0
        num_negative = self.neg_pos_ratio * foreground_idxs.sum(1, keepdim=True)

        negative_loss = cls_loss.clone()
        negative_loss[foreground_idxs] = -float("inf")
        values, idx = negative_loss.sort(1, descending=True)
        background_idxs = idx.sort(1)[1] < num_negative
        N = max(1, num_foreground)
        return {
            "bbox_regression": bbox_loss.sum() / N,
            "classification": (cls_loss[foreground_idxs].sum() +
                               cls_loss[background_idxs].sum()) / N,
        }

    def forward(self, x, targets=None):
        conv_4_3_out = self.features(x)

        conv_4_3_out_scaled = (self.scale_weight.view(1, -1, 1, 1) *
                               torch.nn.functional.normalize(conv_4_3_out))

        conv_5_3_fc_out = self.conv5_3_fc(conv_4_3_out)
        conv8_2_out = self.conv8_2(conv_5_3_fc_out)
        conv9_2_out = self.conv9_2(conv8_2_out)
        conv10_2_out = self.conv10_2(conv9_2_out)
        conv11_2_out = self.conv11_2(conv10_2_out)

        outputs = [
            conv_4_3_out_scaled,  # 38 x 38
            conv_5_3_fc_out,  # 19 x 19
            conv8_2_out,  # 10 x 10
            conv9_2_out,  # 5 x 5
            conv10_2_out,  # 3 x 3
            conv11_2_out,   # 1 x 1
        ]

        cls_logits = []
        bbox_reg_deltas = []
        for i, features in enumerate(outputs):
            cls_feat_i = self.cls_heads[i](features)
            bbox_reg_feat_i = self.bbox_reg_heads[i](features)

            N, _, H, W = cls_feat_i.shape
            cls_feat_i = cls_feat_i.view(N, -1, self.num_classes, H, W)
            cls_feat_i = cls_feat_i.permute(0, 3, 4, 1, 2)
            cls_feat_i = cls_feat_i.reshape(N, -1, self.num_classes)
            cls_logits.append(cls_feat_i)

            N, _, H, W = bbox_reg_feat_i.shape
            bbox_reg_feat_i = bbox_reg_feat_i.view(N, -1, 4, H, W)
            bbox_reg_feat_i = bbox_reg_feat_i.permute(0, 3, 4, 1, 2)
            bbox_reg_feat_i = bbox_reg_feat_i.reshape(N, -1, 4)
            bbox_reg_deltas.append(bbox_reg_feat_i)

        cls_logits = torch.cat(cls_logits, dim=1)  # (B, 8732, num_classes)
        bbox_reg_deltas = torch.cat(bbox_reg_deltas, dim=1)  # (B, 8732, 4)

        default_boxes = generate_default_boxes(outputs, self.aspect_ratios, self.scales)

        losses = {}
        detections = []
        if self.training:
            matched_idxs = []
            for default_boxes_per_image, targets_per_image in zip(default_boxes,
                                                                  targets):
                if targets_per_image["boxes"].numel() == 0:
                    matched_idxs.append(
                        torch.full(
                            (default_boxes_per_image.size(0),), -1,
                            dtype=torch.int64,
                            device=default_boxes_per_image.device
                        )
                    )
                    continue
                iou_matrix = get_iou(targets_per_image["boxes"],
                                     default_boxes_per_image)
                matched_vals, matches = iou_matrix.max(dim=0)

                below_low_threshold = matched_vals < self.iou_threshold
                matches[below_low_threshold] = -1

                _, highest_quality_pred_foreach_gt = iou_matrix.max(dim=1)
                matches[highest_quality_pred_foreach_gt] = torch.arange(
                    highest_quality_pred_foreach_gt.size(0), dtype=torch.int64,
                    device=highest_quality_pred_foreach_gt.device
                )
                matched_idxs.append(matches)
            losses = self.compute_loss(targets, cls_logits, bbox_reg_deltas,
                                       default_boxes, matched_idxs)
        else:
            cls_scores = torch.nn.functional.softmax(cls_logits, dim=-1)
            num_classes = cls_scores.size(-1)

            for bbox_deltas_i, cls_scores_i, default_boxes_i in zip(bbox_reg_deltas,
                                                                    cls_scores,
                                                                    default_boxes):
                boxes = apply_regression_pred_to_default_boxes(bbox_deltas_i,
                                                               default_boxes_i)
                boxes.clamp_(min=0., max=1.)

                pred_boxes = []
                pred_scores = []
                pred_labels = []
                for label in range(1, num_classes):
                    score = cls_scores_i[:, label]

                    keep_idxs = score > self.low_score_threshold
                    score = score[keep_idxs]
                    box = boxes[keep_idxs]

                    score, top_k_idxs = score.topk(min(self.pre_nms_topK, len(score)))
                    box = box[top_k_idxs]

                    pred_boxes.append(box)
                    pred_scores.append(score)
                    pred_labels.append(torch.full_like(score, fill_value=label,
                                                       dtype=torch.int64,
                                                       device=cls_scores.device))

                pred_boxes = torch.cat(pred_boxes, dim=0)
                pred_scores = torch.cat(pred_scores, dim=0)
                pred_labels = torch.cat(pred_labels, dim=0)

                keep_mask = torch.zeros_like(pred_scores, dtype=torch.bool)
                for class_id in torch.unique(pred_labels):
                    curr_indices = torch.where(pred_labels == class_id)[0]
                    curr_keep_idxs = torch.ops.torchvision.nms(pred_boxes[curr_indices],
                                                               pred_scores[curr_indices],
                                                               self.nms_threshold)
                    keep_mask[curr_indices[curr_keep_idxs]] = True
                keep_indices = torch.where(keep_mask)[0]
                post_nms_keep_indices = keep_indices[pred_scores[keep_indices].sort(
                    descending=True)[1]]
                keep = post_nms_keep_indices[:self.detections_per_img]
                pred_boxes, pred_scores, pred_labels = (pred_boxes[keep],
                                                        pred_scores[keep],
                                                        pred_labels[keep])

                detections.append(
                    {
                        "boxes": pred_boxes,
                        "scores": pred_scores,
                        "labels": pred_labels,
                    }
                )
        return losses, detections

**TRAINING**

In [4]:
CocoSSDDataset
collate_fn
SSD

__main__.SSD

In [5]:
# ============================================================
# SSD TRAINING + FINAL EVALUATION
# ============================================================

import glob
import os
import time

import torch
import pandas as pd
from torch.utils.data import DataLoader
from torchvision.ops import box_iou

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from torchmetrics.detection import MeanAveragePrecision


# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    "aspect_ratios": [
        [2],
        [2, 3],
        [2, 3],
        [2, 3],
        [2],
        [2]
    ],

    "scales": [
        0.1,
        0.2,
        0.375,
        0.55,
        0.725,
        0.9
    ],

    "iou_threshold": 0.5,
    "low_score_threshold": 0.01,
    "neg_pos_ratio": 3,

    "pre_nms_topK": 400,
    "nms_threshold": 0.45,
    "detections_per_img": 200,

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    "img_size": 300,
    "batch_size": 16,
    "epochs": 40,

    "lr": 1e-3,
    "momentum": 0.9,
    "weight_decay": 5e-4,

    "warmup_iters": 300,
    "grad_clip": 10.0,

    "num_workers": 2,

    "eval_map_every": 5,

    "save_path": "/kaggle/working/ssd_best.pth"
}


# ============================================================
# FIND DATASET
# ============================================================

def find_dataset_root():

    paths = glob.glob(
        "/kaggle/working/**/train/_annotations.coco.json",
        recursive=True
    )

    if not paths:

        paths = glob.glob(
            "/kaggle/input/**/train/_annotations.coco.json",
            recursive=True
        )

    if not paths:

        raise FileNotFoundError(
            "Roboflow COCO dataset was not found."
        )

    return os.path.dirname(
        os.path.dirname(paths[0])
    )


# ============================================================
# MOVE TARGETS TO DEVICE
# ============================================================

def to_device(targets, device):

    return [
        {
            key: value.to(device)
            for key, value in target.items()
        }
        for target in targets
    ]


# ============================================================
# VALIDATION LOSS
# ============================================================

@torch.no_grad()
def validation_loss(model, loader, device):

    model.train()

    total_loss = 0.0
    batches = 0

    for images, targets in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        targets = to_device(
            targets,
            device
        )

        losses, _ = model(
            images,
            targets
        )

        loss = (
            losses["bbox_regression"]
            + losses["classification"]
        )

        total_loss += loss.item()
        batches += 1

    return total_loss / max(1, batches)


# ============================================================
# mAP EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_map(model, loader, device):

    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox"
    )

    model.eval()

    for images, targets in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        _, detections = model(images)

        predictions = [
            {
                key: value.cpu()
                for key, value in detection.items()
            }
            for detection in detections
        ]

        ground_truth = [
            {
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu()
            }
            for target in targets
        ]

        metric.update(
            predictions,
            ground_truth
        )

    result = metric.compute()

    return {
        "map": result["map"].item(),
        "map_50": result["map_50"].item()
    }


# ============================================================
# FINAL CLASSIFICATION METRICS
# ============================================================

@torch.no_grad()
def evaluate_classification_metrics(
    model,
    loader,
    device,
    iou_threshold=0.5,
    score_threshold=0.01
):

    model.eval()

    y_true = []
    y_pred = []

    for images, targets in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        _, detections = model(images)

        for target, detection in zip(
            targets,
            detections
        ):

            gt_boxes = target["boxes"].cpu()
            gt_labels = target["labels"].cpu()

            pred_boxes = detection["boxes"].cpu()
            pred_labels = detection["labels"].cpu()
            pred_scores = detection["scores"].cpu()

            # ------------------------------------------------
            # SCORE FILTER
            # ------------------------------------------------

            keep = pred_scores >= score_threshold

            pred_boxes = pred_boxes[keep]
            pred_labels = pred_labels[keep]

            matched_gt = set()
            matched_pred = set()

            # ------------------------------------------------
            # MATCH PREDICTIONS TO GROUND TRUTH
            # ------------------------------------------------

            if (
                len(gt_boxes) > 0
                and len(pred_boxes) > 0
            ):

                ious = box_iou(
                    gt_boxes,
                    pred_boxes
                )

                # Match highest IoU first
                matches = []

                for gt_idx in range(len(gt_boxes)):

                    for pred_idx in range(len(pred_boxes)):

                        iou = ious[
                            gt_idx,
                            pred_idx
                        ].item()

                        if iou >= iou_threshold:

                            matches.append(
                                (
                                    iou,
                                    gt_idx,
                                    pred_idx
                                )
                            )

                matches.sort(
                    reverse=True
                )

                for iou, gt_idx, pred_idx in matches:

                    if gt_idx in matched_gt:
                        continue

                    if pred_idx in matched_pred:
                        continue

                    matched_gt.add(gt_idx)
                    matched_pred.add(pred_idx)

                    y_true.append(
                        gt_labels[gt_idx].item()
                    )

                    y_pred.append(
                        pred_labels[pred_idx].item()
                    )

            # ------------------------------------------------
            # FALSE NEGATIVES
            # ------------------------------------------------

            for gt_idx in range(len(gt_boxes)):

                if gt_idx not in matched_gt:

                    y_true.append(
                        gt_labels[gt_idx].item()
                    )

                    # 0 = background / missed detection
                    y_pred.append(0)

            # ------------------------------------------------
            # FALSE POSITIVES
            # ------------------------------------------------

            for pred_idx in range(len(pred_boxes)):

                if pred_idx not in matched_pred:

                    # 0 = background
                    y_true.append(0)

                    y_pred.append(
                        pred_labels[pred_idx].item()
                    )

    # --------------------------------------------------------
    # CALCULATE METRICS
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ============================================================
# TRAINING FUNCTION
# ============================================================

def train(
    root=None,
    config=CONFIG
):

    # ========================================================
    # START TOTAL TIMER
    # ========================================================

    total_start_time = time.time()

    # ========================================================
    # DATASET ROOT
    # ========================================================

    root = root or find_dataset_root()

    # ========================================================
    # DEVICE
    # ========================================================

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print("=" * 70)
    print("SSD TRAINING")
    print("=" * 70)

    print(
        f"Dataset : {root}"
    )

    print(
        f"Device  : {device}"
    )

    if torch.cuda.is_available():

        print(
            f"GPU     : {torch.cuda.get_device_name(0)}"
        )

    # ========================================================
    # DATASETS
    # ========================================================

    train_ds = CocoSSDDataset(
        root,
        "train",
        config["img_size"],
        train=True
    )

    val_ds = CocoSSDDataset(
        root,
        "valid",
        config["img_size"],
        train=False
    )

    test_ds = CocoSSDDataset(
        root,
        "test",
        config["img_size"],
        train=False
    )

    # ========================================================
    # DATASET INFORMATION
    # ========================================================

    print("\n" + "=" * 70)
    print("DATASET INFORMATION")
    print("=" * 70)

    print(
        "Classes:",
        train_ds.class_names
    )

    print(
        f"Train images      : {len(train_ds)}"
    )

    print(
        f"Validation images : {len(val_ds)}"
    )

    print(
        f"Test images       : {len(test_ds)}"
    )

    print("=" * 70)

    # ========================================================
    # DATA LOADERS
    # ========================================================

    train_loader = DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["num_workers"],
        collate_fn=collate_fn,
        pin_memory=True,
        drop_last=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        collate_fn=collate_fn,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        collate_fn=collate_fn,
        pin_memory=True
    )

    # ========================================================
    # MODEL
    # ========================================================

    model = SSD(
        config,
        num_classes=train_ds.num_classes
    ).to(device)

    # ========================================================
    # OPTIMIZER
    # ========================================================

    params = [
        p
        for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.SGD(
        params,
        lr=config["lr"],
        momentum=config["momentum"],
        weight_decay=config["weight_decay"]
    )

    # ========================================================
    # SCHEDULER
    # ========================================================

    epochs = config["epochs"]

    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[
            int(epochs * 0.7),
            int(epochs * 0.9)
        ],
        gamma=0.1
    )

    # ========================================================
    # TRAINING VARIABLES
    # ========================================================

    best_val_loss = float("inf")

    iteration = 0

    history = []

    # ========================================================
    # TRAINING LOOP
    # ========================================================

    for epoch in range(
        1,
        epochs + 1
    ):

        model.train()

        epoch_start_time = time.time()

        total_loc_loss = 0.0
        total_cls_loss = 0.0

        # ----------------------------------------------------
        # BATCH LOOP
        # ----------------------------------------------------

        for images, targets in train_loader:

            # ------------------------------------------------
            # LEARNING RATE WARMUP
            # ------------------------------------------------

            if iteration < config["warmup_iters"]:

                scale = (
                    (iteration + 1)
                    / config["warmup_iters"]
                )

                for group in optimizer.param_groups:

                    group["lr"] = (
                        config["lr"]
                        * scale
                    )

            iteration += 1

            # ------------------------------------------------
            # MOVE DATA
            # ------------------------------------------------

            images = images.to(
                device,
                non_blocking=True
            )

            targets = to_device(
                targets,
                device
            )

            # ------------------------------------------------
            # FORWARD
            # ------------------------------------------------

            losses, _ = model(
                images,
                targets
            )

            loss = (
                losses["bbox_regression"]
                + losses["classification"]
            )

            # ------------------------------------------------
            # BACKWARD
            # ------------------------------------------------

            optimizer.zero_grad()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                params,
                config["grad_clip"]
            )

            optimizer.step()

            # ------------------------------------------------
            # RECORD LOSSES
            # ------------------------------------------------

            total_loc_loss += (
                losses[
                    "bbox_regression"
                ].item()
            )

            total_cls_loss += (
                losses[
                    "classification"
                ].item()
            )

        # ====================================================
        # END OF EPOCH
        # ====================================================

        scheduler.step()

        batches = len(train_loader)

        avg_loc_loss = (
            total_loc_loss / batches
        )

        avg_cls_loss = (
            total_cls_loss / batches
        )

        epoch_time = (
            time.time()
            - epoch_start_time
        )

        # ====================================================
        # VALIDATION
        # ====================================================

        val_loss = validation_loss(
            model,
            val_loader,
            device
        )

        # ====================================================
        # EPOCH OUTPUT
        # ====================================================

        message = (
            f"Epoch {epoch:03d}/{epochs} "
            f"| loc {avg_loc_loss:.4f} "
            f"| cls {avg_cls_loss:.4f} "
            f"| val {val_loss:.4f} "
            f"| lr {optimizer.param_groups[0]['lr']:.6f} "
            f"| {epoch_time:.0f}s"
        )

        # ====================================================
        # VALIDATION mAP
        # ====================================================

        if (
            epoch % config["eval_map_every"] == 0
            or epoch == epochs
        ):

            map_result = evaluate_map(
                model,
                val_loader,
                device
            )

            message += (
                f" | mAP {map_result['map']:.4f}"
                f" | mAP50 {map_result['map_50']:.4f}"
            )

        print(message)

        # ====================================================
        # SAVE HISTORY
        # ====================================================

        history.append({

            "epoch": epoch,

            "loc_loss":
                avg_loc_loss,

            "cls_loss":
                avg_cls_loss,

            "val_loss":
                val_loss,

            "epoch_time":
                epoch_time
        })

        # ====================================================
        # SAVE BEST MODEL
        # ====================================================

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            torch.save(
                {
                    "model":
                        model.state_dict(),

                    "class_names":
                        train_ds.class_names,

                    "config":
                        config
                },
                config["save_path"]
            )

            print(
                f"  -> Best model saved "
                f"(val loss: {best_val_loss:.4f})"
            )

    # ========================================================
    # TOTAL TRAINING TIME
    # ========================================================

    total_training_time = (
        time.time()
        - total_start_time
    )

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    checkpoint = torch.load(
        config["save_path"],
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model"]
    )

    model.eval()

    print("\n" + "=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)

    print(
        f"Training time: "
        f"{total_training_time:.2f} seconds"
    )

    print(
        f"Training time: "
        f"{total_training_time / 60:.2f} minutes"
    )

    print(
        f"Training time: "
        f"{total_training_time / 3600:.2f} hours"
    )

    # ========================================================
    # FINAL TEST EVALUATION
    # ========================================================

    print("\n" + "=" * 70)
    print("FINAL TEST SET EVALUATION")
    print("=" * 70)

    print("Evaluating mAP...")

    map_result = evaluate_map(
        model,
        test_loader,
        device
    )

    print("Evaluating Accuracy, Precision, Recall and F1...")

    classification_result = evaluate_classification_metrics(
        model,
        test_loader,
        device,
        iou_threshold=config["iou_threshold"],
        score_threshold=config["low_score_threshold"]
    )

    # ========================================================
    # COMBINE RESULTS
    # ========================================================

    accuracy = classification_result["accuracy"]
    precision = classification_result["precision"]
    recall = classification_result["recall"]
    f1 = classification_result["f1"]

    map_score = map_result["map"]
    map50_score = map_result["map_50"]

    # ========================================================
    # METRICS TABLE
    # ========================================================

    metrics_table = pd.DataFrame({

        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "mAP",
            "mAP@0.50",
            "Training Time"
        ],

        "Score": [
            accuracy,
            precision,
            recall,
            f1,
            map_score,
            map50_score,
            total_training_time
        ],

        "Percentage": [
            f"{accuracy * 100:.2f}%",
            f"{precision * 100:.2f}%",
            f"{recall * 100:.2f}%",
            f"{f1 * 100:.2f}%",
            f"{map_score * 100:.2f}%",
            f"{map50_score * 100:.2f}%",
            f"{total_training_time:.2f} seconds"
        ]
    })

    # ========================================================
    # DISPLAY FINAL TABLE
    # ========================================================

    print("\n" + "=" * 70)
    print("FINAL MODEL PERFORMANCE")
    print("=" * 70)

    display(
        metrics_table.style.hide(
            axis="index"
        )
    )

    print("=" * 70)

    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return (
        model,
        history,
        total_training_time,
        metrics_table
    )


# ============================================================
# START TRAINING
# ============================================================

model, history, total_training_time, metrics_table = train()

SSD TRAINING
Dataset : /kaggle/working/final-aimedisina-7
Device  : cuda
GPU     : Tesla T4

DATASET INFORMATION
Classes: ['__background__', 'Authentic -Medicol Advance', 'Authentic-Alaxan', 'Authentic-Bioflu', 'Authentic-Decolgen', 'Authentic-Neozep Forte', 'Authentic_Biogesic', 'Back', 'Counterfeit-Alaxan', 'Counterfeit-Bioflu', 'Counterfeit-Decolgen', 'Counterfeit-Medicol Advance', 'Counterfeit-Neozep Forte', 'Counterfeit_Biogesic', 'Front']
Train images      : 5184
Validation images : 111
Test images       : 332
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 196MB/s]


Epoch 001/40 | loc 1.3836 | cls 6.9984 | val 3.4478 | lr 0.001000 | 179s
  -> Best model saved (val loss: 3.4478)
Epoch 002/40 | loc 0.5820 | cls 2.1846 | val 2.7363 | lr 0.001000 | 184s
  -> Best model saved (val loss: 2.7363)
Epoch 003/40 | loc 0.4035 | cls 1.6889 | val 2.1104 | lr 0.001000 | 184s
  -> Best model saved (val loss: 2.1104)
Epoch 004/40 | loc 0.3305 | cls 1.4330 | val 1.9671 | lr 0.001000 | 184s
  -> Best model saved (val loss: 1.9671)
Epoch 005/40 | loc 0.2860 | cls 1.2693 | val 1.8069 | lr 0.001000 | 184s | mAP 0.6599 | mAP50 0.8202
  -> Best model saved (val loss: 1.8069)
Epoch 006/40 | loc 0.2515 | cls 1.1548 | val 1.6692 | lr 0.001000 | 184s
  -> Best model saved (val loss: 1.6692)
Epoch 007/40 | loc 0.2220 | cls 1.0759 | val 1.5737 | lr 0.001000 | 184s
  -> Best model saved (val loss: 1.5737)
Epoch 008/40 | loc 0.1997 | cls 0.9974 | val 1.5765 | lr 0.001000 | 185s
Epoch 009/40 | loc 0.1816 | cls 0.9522 | val 1.5973 | lr 0.001000 | 184s
Epoch 010/40 | loc 0.1663 | 

Metric,Score,Percentage
Accuracy,0.070968,7.10%
Precision,0.034378,3.44%
Recall,0.070968,7.10%
F1-Score,0.043818,4.38%
mAP,0.811092,81.11%
mAP@0.50,0.919057,91.91%
Training Time,7472.782208,7472.78 seconds
